# Redline on a free GPU (Kaggle or Colab)

Runs a real **before vs after** safety regression check on two open models, with nothing installed on your laptop:
1. **Ollama** serves the models on the notebook's GPU.
2. Redline runs the **identical cases** against both: jailbreaks, harmful requests, over-refusal, hallucination, and **prompt injection graded in code** from the agent's tool calls.
3. You get a terminal summary and a self-contained **HTML report** to download.

**Kaggle setup (once):**
1. Verify your phone number (Settings → Phone verification). This unlocks GPUs and internet.
2. In this notebook: *Settings → Accelerator → **GPU T4 x2*** and *Settings → **Internet on***.
3. Optional, for a real LLM judge: *Add-ons → Secrets*, add **`GROQ_API_KEY`** (free key at console.groq.com, no card needed).

Kaggle gives about 30 free GPU hours a week. One run of this notebook takes about 15–30 minutes.

**Data rule:** free hosted judges are for public test cases only, never customer data.

In [ ]:
# ── 1. Settings: edit these ──────────────────────────────────────────────
# A base model and a fine-tune of it (same family), so a difference = the fine-tune.
BEFORE_MODEL = "llama3.1:8b"   # Meta Llama 3.1 8B Instruct (the base)
AFTER_MODEL  = "hermes3:8b"    # Nous Hermes 3: a fine-tune of Llama 3.1 8B
CASES        = 40              # 30+ for a rated (non-Provisional) tier
PACKS        = "core,financial_services"
USE_GROQ_JUDGE = False         # False = free keyword judge (rough); True = LLM judge via your GROQ_API_KEY secret
GROQ_JUDGE_MODEL = "groq/llama-3.3-70b-versatile"  # check console.groq.com/docs/models for current names
REDLINE_GIT = "https://github.com/kush0o7/Redline-Agent-Safety-Eval.git"
REDLINE_REF = "main"

In [ ]:
# ── 2. Get the Redline code ──────────────────────────────────────────────
# Prefers code you uploaded as a Kaggle dataset (the zip, or the folder Kaggle unzips it into).
# Falls back to cloning GitHub, which may be older than your local copy: the version line below tells you.
import os, shutil, subprocess, zipfile
os.chdir("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
shutil.rmtree("redline", ignore_errors=True)
MARK = "backend/app/evals/data/jbb_behaviors.json"   # only present in the fixed code
src = None
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    if root.endswith("/backend") and "pyproject.toml" in files:
        src = ("folder", root[: -len("/backend")])
    for f in files:
        if f.endswith(".zip") and any(n.endswith("backend/pyproject.toml") for n in zipfile.ZipFile(os.path.join(root, f)).namelist()):
            src = ("zip", os.path.join(root, f))
    if src: break
if src and src[0] == "zip":
    zipfile.ZipFile(src[1]).extractall("redline"); print("Using uploaded zip:", src[1])
elif src:
    shutil.copytree(src[1], "redline"); print("Using uploaded folder:", src[1])
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", REDLINE_REF, REDLINE_GIT, "redline"], check=True)
    print("No uploaded code found under /kaggle/input: cloned", REDLINE_GIT, REDLINE_REF)
REDLINE = os.path.abspath("redline")
print("Code version:", "NEW (fixed)" if os.path.exists(f"{REDLINE}/{MARK}") else "OLD: upload the latest redline-code.zip as a new dataset version")


In [ ]:
# ── 3. Install and start Ollama on the GPU, then download the two models ──
import subprocess, time, urllib.request
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama is up")
for m in (BEFORE_MODEL, AFTER_MODEL):
    print("pulling", m); subprocess.run(["ollama", "pull", m], check=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

In [ ]:
# ── 4. Install Redline's dependencies ─────────────────────────────────────
# Installs only what Redline needs (never jailbreakbench: it can't install on Python 3.13).
# Redline itself isn't pip-installed: scripts/demo.py loads it from the folder.
import subprocess, sys, tomllib, os
meta = tomllib.load(open(f"{REDLINE}/backend/pyproject.toml", "rb"))
deps = meta["project"]["dependencies"]
has_jbb = any(d.startswith("jailbreakbench") for d in deps)
print("Code location:", REDLINE)
print("Code version:", "OLD (still lists jailbreakbench)" if has_jbb else "NEW (fixed)")
print("Bundled JailbreakBench data present:", os.path.exists(f"{REDLINE}/backend/app/evals/data/jbb_behaviors.json"))
deps = [d for d in deps if not d.startswith("jailbreakbench")]
r = subprocess.run([sys.executable, "-m", "pip", "install", *deps], capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    raise SystemExit("pip failed: see the output above")
print("installed", len(deps), "packages")


In [ ]:
# ── 5. Run the before → after regression check ───────────────────────────
import os, subprocess, sys
env = dict(os.environ,
    DEMO_BEFORE_MODEL=BEFORE_MODEL, DEMO_AFTER_MODEL=AFTER_MODEL,
    DEMO_CASES=str(CASES), DEMO_PACKS=PACKS, DEMO_NO_OPEN="1",
    DEMO_OUT_DIR=os.path.abspath("reports"),
    LLM_PROVIDER="ollama", OLLAMA_BASE_URL="http://127.0.0.1:11434",
)
if USE_GROQ_JUDGE:
    try:
        from kaggle_secrets import UserSecretsClient
        env["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    except Exception:
        from google.colab import userdata  # Colab: add the key under the key icon (Secrets)
        env["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    env["JUDGE_MODEL"] = GROQ_JUDGE_MODEL   # Redline paces Groq to >= 10 s per case (free-tier limits)
else:
    env["DEMO_FAKE_JUDGE"] = "1"            # keyword judge: injection is still graded in code
    env["EVAL_PACING_SECONDS"] = "0"
proc = subprocess.run([sys.executable, f"{REDLINE}/scripts/demo.py"], env=env, cwd=REDLINE,
                      capture_output=True, text=True)
print(proc.stdout[-15000:])
if proc.returncode:
    print("STDERR:", proc.stderr[-5000:])
    print("If a model 'does not support tools', pick another model with tool support (ollama.com/search?c=tools).")

In [ ]:
# ── 6. The HTML report (download it from the Output panel → reports/) ──────
import glob
reports = sorted(glob.glob("reports/redline-demo-comparison-*.html"))
print("\n".join(reports) or "no report written; check the output above")
if reports:
    from IPython.display import HTML, display
    display(HTML(f'<p>Report: <code>{reports[-1]}</code>. Open the Output panel on the right to download it.</p>'))

## Reading the result
- **Release decision** (PASS/FAIL) and **Significance**: *significant (p<0.05)* means the change is unlikely to be noise. McNemar's exact test runs on the cases that flipped.
- **decided by code**: injection cases graded from the agent's actual tool calls. These are solid even with the free keyword judge.
- With the **keyword judge**, harmful-content verdicts are rough. Read the failed cases yourself, or rerun with `USE_GROQ_JUDGE = True`.
- **More pairs to try** (base → fine-tune, all ≤ 8B so they fit a T4): `qwen2.5:7b` → `qwen2.5-coder:7b`, or `llama3.2:3b` → `hermes3:3b`. Check each model's tool support on ollama.com first.